# PROC INBREED를 이용한 전력 변압기 설계군의 공유 설계 유산 정량화


## 요약

한 전력망 운영사의 변전소 변압기는 순차적인 설계 세대에 걸쳐 개발되며, 각 신모델은 두 개의 선행 설계로부터 파생됩니다. 이 노트북은 이러한 엔지니어링 계보를 하나의 계통도(pedigree)로 취급하고, **PROC INBREED**를 사용하여 근친 계수와 가산적 연관성(공분산) 계수를 계산함으로써 두 자산이 얼마나 많은 설계 유산을 공유하는지 정량화합니다.

계통도는 구조를 해석하기 쉽도록 구성되어 있습니다. 현재 세대(3세대) 모델 4종 중 2종은 **형제 관계(sibling)**인 한 쌍의 선행 설계에서 파생되어 유산이 집중되어 있는 반면, 나머지 2종은 서로 겹치지 않는 계통을 사용합니다. 이 절차는 이 구조를 정확히 재현합니다. 형제 설계에서 파생된 두 모델(`G3_T1`, `G3_T3`)은 각각 근친 계수 **F = 0.25**를 가지며, 나머지 두 모델(`G3_T2`, `G3_T4`)은 **F = 0**입니다. 전체 설계군의 평균 근친 계수는 **0.0417**입니다. 가산적 연관성 행렬을 보면, 현재 세대 모델 중 연관성이 가장 낮은 쌍은 **`G3_T1`과 `G3_T3`(연관성 = 0)**입니다 — 이 둘은 공통 조상이 전혀 없어 가장 강력한 이중화(N-1) 조합을 이루며, `G3_T1` 자체가 가장 근친도가 높은 설계 중 하나라는 점에서 특히 중요합니다.


## 데이터 소스

| 데이터셋 | 행 수 | 주요 변수 | 설명 |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | 서로 겹치지 않는 세 설계 세대(4개의 창립 플랫폼, 4개의 2세대 파생 설계, 4개의 현재 세대 모델)에 걸친, 작고 결정론적인 변전소 변압기 설계 계통도입니다. 창립 설계가 아닌 각 설계는 자신이 파생된 두 개의 서로 다른 선행 설계(`Parent1`, `Parent2`)를 기록합니다. `Sex`는 주도 설계 계통을 표시합니다(M = 기계/철심 계통, F = 전기/권선 계통). 이 계통도는 무작위가 아니라 직접 지정한 것이므로, 연관성 구조를 사전에 알 수 있고 절차의 출력과 대조하여 검증할 수 있습니다. |


# 전력 변압기 설계군의 공유 설계 유산 정량화

## 전력회사가 "근친도"에 관심을 갖는 이유

송배전 운영사는 수백 대의 변전소 전력 변압기를 운용합니다. 엔지니어링 비용과 검증 부담을 관리하기 위해 제조사는 변압기를 매번 처음부터 설계하는 경우가 거의 없습니다 — 모든 신모델은 하나 또는 두 개의 선행 모델로부터 핵심 형상, 권선 토폴로지, 절연 시스템, 부싱 설계를 **물려받습니다**. 여러 조달 주기를 거치면서 이는 진정한 *엔지니어링 계보*, 즉 설계의 계통도를 만들어냅니다.

이렇게 공유된 유산은 숨겨진 신뢰성 위험입니다. 동일한 부하를 보호하는 두 대의 변압기(이중화 **N-1** 쌍)가 동일한 조상 설계에서 파생되었다면, 잠재적 설계 결함 — 공진 모드, 절연 열화 메커니즘, 부싱 섬락 경로 — 이 **두 대 모두**에 존재할 가능성이 큽니다. 그러면 단 하나의 근본 원인이 이중화 쌍을 동시에 무력화시킬 수 있습니다: 이른바 *공통 모드 고장*입니다.

**PROC INBREED**는 바로 이러한 종류의 공유 계통을 정량화하기 위해 만들어졌습니다. 그 기원은 동식물 육종에 있지만, 그 수학적 기반인 라이트(Wright)의 가산적 연관성 재귀식은 분야에 관계없이 적용됩니다 — 두 개체가 공통 조상을 통해 공유하는 설계 유산의 비율을 측정합니다. 유전학 용어를 자산 엔지니어링에 대응시키면 다음과 같습니다.

| INBREED 개념 | 전력회사 관점의 해석 |
|---|---|
| 개체(Individual) | 변압기 설계 / 모델 |
| 부모(sire/dam) | 파생의 근원이 된 선행 설계 |
| 세대(CLASS) | 조달 / 설계 주기 |
| 근친 계수 *F* | 한 설계 내에서 유산이 자기 유사적으로 집중된 정도 |
| 공분산 / 연관성 계수 | 두 설계 간에 공유된 설계 유산 |
| 최소 연관 쌍 | N-1 복원력을 위한 최적의 이중화 조합 |


## 1단계 — 설계 계통도 지정

연관성 구조가 명확하도록 `DesignLineage`를 직접 입력합니다. 서로 겹치지 않는 **세 개의 설계 세대**가 있습니다.

- **1세대** — 선행 설계가 없는(부모 칸이 비어 있는) 4개의 창립 플랫폼 설계(`G1_P1`~`G1_P4`).
- **2세대** — 각각 서로 **다른** 두 개의 1세대 플랫폼으로부터 설계된 4개의 파생 설계(`G2_D1`~`G2_D4`). `G2_D1`과 `G2_D2`는 *친형제(full sibling)* 관계입니다(둘 다 `G1_P1` x `G1_P2`에서 파생). `G2_D3`과 `G2_D4`도 친형제입니다(둘 다 `G1_P3` x `G1_P4`에서 파생).
- **3세대** — 4개의 현재 세대 모델(`G3_T1`~`G3_T4`). `G3_T1`은 형제 쌍 `G2_D1` x `G2_D2`에서, `G3_T3`은 형제 쌍 `G2_D3` x `G2_D4`에서 만들어져 두 설계 모두 유산이 집중되어 있습니다. `G3_T2`와 `G3_T4`는 각각 서로 겹치지 않는 두 계통을 교차시킵니다.

`ID`, `Parent1`, `Parent2` 열이 계통 정보를 담고 있으며, `Sex`는 어느 엔지니어링 계통이 설계를 주도했는지를 기록합니다. 이어지는 짧은 DATA 스텝은 PROC INBREED가 기대하는 대로 창립 플랫폼의 부모 칸이 비도록 자리표시자 `.`을 공백으로 바꿉니다.


In [1]:
데이터 DesignLineage;
   길이 ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE 자료 truncover;
   입력 Generation ID $ Parent1 $ Parent2 $ Sex $;
   자료;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
실행;

/* 창립 플랫폼은 선행 설계가 없으므로 자리표시자 점(.)을 공백으로 바꿈 */
데이터 DesignLineage;
   설정 DesignLineage;
   만약 Parent1 = '.' 이면 Parent1 = ' ';
   만약 Parent2 = '.' 이면 Parent2 = ' ';
실행;

제목 '변압기 설계 계통도';
처리 인쇄 데이터=DesignLineage noobs;
   변수 Generation ID Parent1 Parent2 Sex;
   라벨 Generation='세대' ID='설계 ID' Parent1='상위설계 1' Parent2='상위설계 2' Sex='리드 계통(M/F)';
실행;


                                                       변압기 설계 계통도                                                       


    세대      설계 ID          상위설계 1          상위설계 2          리드 계통(M/F)
------  ---------  --------------  --------------  ------------------
     1  G1_P1                                      M
     1  G1_P2                                      M
     1  G1_P3                                      M
     1  G1_P4                                      F
     2  G2_D1      G1_P1           G1_P2           M
     2  G2_D2      G1_P1           G1_P2           F
     2  G2_D3      G1_P3           G1_P4           F
     2  G2_D4      G1_P3           G1_P4           M
     3  G3_T1      G2_D1           G2_D2           M
     3  G3_T2      G2_D1           G2_D3           F
     3  G3_T3      G2_D3           G2_D4           F
     3  G3_T4      G2_D2           G2_D4           M




NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to 변압기 설계 계통도.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## 2단계 — 설계 세대별 근친 계수

`CLASS` 문에 `Generation`을 지정하여 PROC INBREED를 **다세대 모드**로 실행하면 세대별 계통 요약이 출력됩니다. `VAR` 문은 개체, 첫 번째 선행 설계, 두 번째 선행 설계 순서로 세 개의 계통 열을 지정합니다.

- **IND** 옵션은 각 설계의 근친 계수 *F*를 출력합니다 — 이는 해당 설계 자체의 유산이 얼마나 집중되어 있는지를 나타냅니다. 서로 밀접하게 연관된 두 선행 설계로부터 만들어진 설계는 *F*가 높습니다.
- **MATRIX** 옵션은 전체 가산적 연관성 행렬을 출력하여 쌍별 유산을 직접 확인할 수 있게 합니다.
- **AVERAGE** 옵션은 설계군 전체의 평균 근친 계수를 추가로 출력합니다 — 설계 다양성을 하나의 값으로 요약한 것입니다.

값이 0에 가까울수록 독립적인 계통을 의미하며, 설계의 두 선행 설계가 서로 더 밀접하게 연관될수록 *F*가 커집니다.


In [2]:
제목 '설계 세대별 근친 계수';
처리 inbreed 데이터=DesignLineage ind average MATRIX;
   변수 ID Parent1 Parent2;
   분류 Generation;
   라벨 ID='설계 ID' Parent1='상위설계 1' Parent2='상위설계 2' Generation='세대';
실행;


                                                      설계 세대별 근친 계수                                                      


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by 세대/Class
--------------------------------------
세대 1            Members = 4
세대 2            Members = 4
세대 3            Members = 4

Inbreeding Coefficients (세대 1)
--------------------------------------
설계 ID               F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (세대 2)
--------------------------------------
설계 ID               F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (세대 3)
-------------------------------


NOTE: Option TITLE changed to 설계 세대별 근친 계수.
NOTE: PROC INBREED data=DesignLineage



## 3단계 — 후속 위험 점수화를 위한 공분산 계수 저장

근친도 보기를 **COVAR** 옵션으로 바꾸면 동일한 관계를 자산관리팀이 이중화 위험 점수에 입력할 수 있는 형태인 **공분산(가산적 연관성) 계수**로 보고합니다. **OUTCOV=** 옵션은 전체 행렬을 데이터셋(`DesignCovar`)으로 저장하며, 여기서 `Col1`~`Col12` 열은 각 설계와 다른 모든 설계 사이의 연관성(계통 순서대로)을 담고 있어 변전소 배정 정보와 바로 조인할 수 있습니다.

저장된 행렬이 리스팅과 함께 보이도록 출력 데이터셋을 출력합니다.


In [3]:
제목 '공분산(연관성) 계수';
처리 inbreed 데이터=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   변수 ID Parent1 Parent2;
   분류 Generation;
   라벨 ID='설계 ID' Parent1='상위설계 1' Parent2='상위설계 2' Generation='세대';
실행;

제목 'OUTCOV= 위험 점수화용 저장된 연관성 행렬';
처리 인쇄 데이터=DesignCovar noobs;
실행;


                                                      공분산(연관성) 계수                                                       


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by 세대/Class
--------------------------------------
세대 1            Members = 4
세대 2            Members = 4
세대 3            Members = 4

Inbreeding Coefficients (세대 1)
--------------------------------------
설계 ID               F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (세대 1)
--------------------------------------------------
설계 ID                  G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.0000
G1_P4                0.0000   0.0000   0.0000


NOTE: Option TITLE changed to 공분산(연관성) 계수.
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to OUTCOV= 위험 점수화용 저장된 연관성 행렬.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## 4단계 — 이중화(N-1) 설치를 위한 최소 연관 조합

저장된 `DesignCovar` 행렬은 이중화 연구에 정확히 필요한 자료입니다. 설계 *i*와 설계 *j*의 연관성은 *i*행, `Col`*j*열에 있습니다(열은 계통 순서대로 배열되어 있어 `Col9`~`Col12`가 네 개의 현재 세대 모델 `G3_T1`~`G3_T4`에 대응합니다).

`DesignCovar`에서 현재 세대에 해당하는 네 개의 행을 읽어, 가능한 모든 순서 없는 현재 세대 쌍을 구성하고 각 쌍에 연관성 계수를 붙인 다음 연관성이 낮은 순으로 정렬합니다. 목표는 공유 유산이 **가장 낮은** 이중화 쌍을 선택하는 것입니다 — 그래야 하나의 유전된 설계 결함이 N-1 보호 위치의 양쪽을 동시에 무력화시킬 가능성을 최소화할 수 있습니다.


In [4]:
/* DesignCovar에서 현재 세대에 해당하는 네 개의 행을 추출합니다.
   Col9..Col12는 G3_T1..G3_T4에 대한 연관성입니다(계통 열 순서).
   가능한 모든 순서 없는 쌍을 구성합니다. */
데이터 Pairs;
   설정 DesignCovar;
   조건 ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   길이 DesignA $12 DesignB $12;
   배열 g3{4} Col9 Col10 Col11 Col12;
   배열 nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   반복 j = 1 까지 4;
      DesignB = nm{j};
      만약 DesignA < DesignB 이면 반복;
         Relationship = g3{j};
         출력;
      종료;
   종료;
   유지 DesignA DesignB Relationship;
실행;

처리 정렬 데이터=Pairs; 기준 Relationship; 실행;

제목 '이중화(N-1) 후보 조합, 연관성 낮은 순';
처리 인쇄 데이터=Pairs noobs;
   변수 DesignA DesignB Relationship;
   라벨 DesignA='설계 A' DesignB='설계 B' Relationship='연관성 계수';
실행;
제목;


                                                이중화(N-1) 후보 조합, 연관성 낮은 순                                                


    설계 A      설계 B            연관성 계수
--------  --------  ----------------
G3_T1     G3_T3                    0
G3_T2     G3_T4                 0.25
G3_T1     G3_T2                0.375
G3_T1     G3_T4                0.375
G3_T2     G3_T3                0.375
G3_T3     G3_T4                0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to 이중화(N-1) 후보 조합, 연관성 낮은 순.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## 결과 해석

**근친 계수(2단계).** 모든 1세대 창립 플랫폼과 모든 2세대 파생 설계는 *F* = 0으로 나타납니다 — 설계상 두 선행 설계가 서로 연관된 경우가 없기 때문입니다. 3세대 모델 중 두 개가 *F* = 0.25로 나타납니다: `G3_T1`(형제 쌍 `G2_D1` x `G2_D2`에서 제작)과 `G3_T3`(형제 쌍 `G2_D3` x `G2_D4`에서 제작)입니다. 이들의 선행 설계는 *동일한* 창립 플랫폼 쌍으로 거슬러 올라가므로 유산이 집중되어 있습니다. 신뢰성 관점에서 이들은 단일 유전 결함에 가장 취약한 설계이며, 추가적인 독립 검증 시험이 필요합니다. 서로 겹치지 않는 두 계통을 교차시킨 `G3_T2`와 `G3_T4`는 *F* = 0입니다.

**연관성 행렬(3단계).** 대각선 밖의 값들은 쌍별 공유 유산을 정량화합니다. 형제 관계인 두 2세대 쌍은 서로에 대해 각각 0.50의 연관성을 보이는 반면(`G2_D1`-`G2_D2`, `G2_D3`-`G2_D4`), 반대 계통에서 나온 파생 설계는 0.00입니다. 근친 관계인 현재 세대 설계는 대각선에서 1.25(= 1 + *F*)의 자기 연관성을 보입니다. `DesignCovar` 데이터셋은 변전소 배정 정보와 조인할 수 있도록 전체 행렬을 제공합니다.

**최소 연관 조합(4단계).** 모든 현재 세대 쌍을 연관성 기준으로 정렬하면 **`G3_T1`과 `G3_T3`이 연관성 0.00으로 1위**에 오릅니다 — 이 둘은 완전히 겹치지 않는 조상 플랫폼에서 파생되어 공유하는 설계 유산이 전혀 없습니다. 이는 가장 강력한 이중화 조합이며, `G3_T1` 자체가 가장 근친도가 높은 두 설계 중 하나라는 점에서 특히 가치가 있습니다 — 완전히 무관한 짝과 짝지음으로써 집중된 유산의 위험을 상쇄할 수 있습니다. 그다음으로 좋은 조합은 연관성 0.25인 `G3_T2`와 `G3_T4`이며, 나머지 조합들은 0.375입니다. 2단계에서 AVERAGE 옵션으로 출력된 설계군의 평균 근친 계수 **0.0417**은 전체 설계 다양성을 요약합니다. 조달 시에는 N-1 위치에 연관성이 가장 낮은 쌍을 우선하고, 장기적으로는 조상 기반을 넓혀야 합니다 — 이는 유전적 병목현상을 피하는 것과 같은 자산 엔지니어링적 원리입니다.

**유의사항.** 이는 예시를 위한 합성 데이터이며, 실제 연구에서는 제조사의 실제 설계 개정 기록으로 계통도를 구성하고 조달 의사결정에 반영하기 전에 연관성 점수를 과거 공통 모드 정전 사례와 대조하여 검증해야 합니다.
